<div style="border:2px solid #B3241C; background:#FDECEA; border-radius:8px; padding:16px 20px;">
  <div style="color:#B3241C; font-size:26px; font-weight:800; line-height:1.25; margin-bottom:8px;">
    STOP &mdash; do this first
  </div>
  <div style="font-size:19px; color:#1F2328; line-height:1.5;">
    <b>File &rarr; Save a copy in Drive</b>, then work in <b>your</b> copy.<br>
    If you skip this step, nothing you type here will be saved.
  </div>
</div>

# K513 Final Project · Notebook 3 — Predicting late delivery

**The decision.** Olist's fulfillment team can flag **500 orders** in the test period for expediting
or a proactive message to the customer. Your model decides which 500. The prediction is made **at the
moment the order is placed** — so the model may only use what is known at that moment.

**How this notebook works**

- It runs on the provided model-ready table, so you can start right away.
- Cells with `____` are yours to complete. **📊 Slide N** tells you where each answer goes.
- The four questions frame everything: unit of analysis, target, classification or regression,
  baseline.

## 0 · Load the table

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (confusion_matrix, accuracy_score, recall_score,
                             precision_score, roc_auc_score)

RANDOM_SEED = 42

pd.set_option('display.precision', 3)
pd.set_option('display.max_columns', 40)

BASE = "https://raw.githubusercontent.com/jl-uscn/k513-data/main/FP"
DATES = ['order_purchase_timestamp', 'order_estimated_delivery_date',
         'order_delivered_carrier_date', 'order_delivered_customer_date']

df = pd.read_csv(f"{BASE}/olist_model_ready.csv.gz", parse_dates=DATES)
print(df.shape)

---
## 1 · The prediction question

**Q1.1** Answer the four questions. What is one row? What is the target, and which class is
**positive**? Classification or regression? What is the base rate, and what does the baseline
(always predicting the majority class) do?

In [ ]:
print("one row = one delivered order;", len(df), "rows")
print(df['____'].value_counts())
print(f"base rate: {df['late'].mean():.3f}")

In [ ]:
# print out the column names. Talk with the team about their meanings. Other students are working on the raw data or
# the data understanding. They might have insights about the features.
print(df.columns)

> 📊 **Slide 27** — the four questions, answered, and the moment of prediction.

---
## 2 · Leakage — which columns can the model not use?

A column can be a feature only if it has a value **when the order is placed**. A column filled in
later — or calculated from one — makes the model look excellent in this notebook and useless in
practice, because at prediction time it is blank.

**Q2.1** Go through every column. List the ones the model must **not** use, each with its reason.
Identifiers are also excluded, for a different reason: an ID number is a label, not an attribute.

Several of these columns appear in Notebook 2 as related to late delivery. For each one, note the
Notebook 2 slide where it showed up — ask the teammate who did Notebook 2.

In [ ]:
IDENTIFIERS = ['order_id', 'customer_unique_id', 'seller_id']

NOT_KNOWN_AT_PURCHASE = ['order_delivered_carrier_date', ____]

print(len(IDENTIFIERS) + len(NOT_KNOWN_AT_PURCHASE), "columns excluded")

> 📊 **Slide 28** — the excluded columns, each with the reason, when it becomes known, and the Notebook 2
> slide where it showed up as related to late delivery (if it did).

---
## 3 · Features

**Q3.1** These are the features: the columns known at purchase. `order_purchase_timestamp` and `order_estimated_delivery_date` are
replaced by `n_days_promised`; city columns have too many values to be useful, so the state
columns stay and the city columns are removed. Check that no excluded column slipped in.

These features are not chosen blindly: Notebook 2 already looked at most of them. For each group of
features, find the Notebook 2 slide whose finding supports including it.

In [ ]:
CONTINUOUS = ['n_items', 'n_diff_products', ____] # in total 12 columns
CATEGORICAL = ['customer_state', 'seller_state', 'payment_type', ____,
               'same_state', ____]

leaked = set(CONTINUOUS + CATEGORICAL) & set(IDENTIFIERS + NOT_KNOWN_AT_PURCHASE)
assert not leaked, f"remove these from the features: {leaked}"
print(len(CONTINUOUS), "continuous +", len(CATEGORICAL), "categorical features")

Four orders have no product dimensions (Notebook 2, Q1.2). The models cannot use a row with a
missing value, so those four rows are dropped.

In [ ]:
model_df = df.dropna(subset=CONTINUOUS + CATEGORICAL)
print("rows:", len(df), "->", len(model_df))

X = model_df[CONTINUOUS + CATEGORICAL]
y = model_df['late']

> 📊 **Slide 29** — the features in plain English, grouped (what was ordered, where it goes, how it
> was paid, what was promised), the Notebook 2 slide that supports each group, and the four dropped rows.

---
## 4 · Split, and the baseline

**Q4.1** Hold out 25% of the orders as the test set. `stratify=y` keeps the share of late orders
the same in both parts.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=____, random_state=RANDOM_SEED, stratify=____)

print(f"train {len(X_train)}   test {len(X_test)}")
print(f"late — train {y_train.mean():.3f}   test {y_test.mean():.3f}")

**Q4.2** The baseline always predicts the majority class. What is its accuracy, and how many late
orders does it catch? If the team flagged 500 test orders **at random**, how many late orders would
they expect to catch?

In [ ]:
baseline = DummyClassifier(strategy='____').fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)

print(f"baseline accuracy: {accuracy_score(y_test, y_pred_baseline):.3f}")
print("late orders caught:", int(((y_pred_baseline == 1) & (y_test == 1)).sum()))
print(f"500 at random would catch about {500 * y_test.mean():.0f} of {y_test.sum()} late orders")

> 📊 **Slide 30** — the baseline's accuracy, the late orders it catches, and what 500 random flags
> would catch. Why is high accuracy not good news here?

---
## 5 · Two models

**Q5.1** Build a logistic regression and a decision tree. The logistic regression needs its
continuous columns scaled; the tree does not. The tree's settings are fixed in advance — do not tune
them by looking at test scores.

In [ ]:
logistic_pre = ColumnTransformer([
    ('cat', OneHotEncoder(drop='if_binary', handle_unknown='ignore'), CATEGORICAL),
    ('cont', ____, CONTINUOUS)])

logistic_model = Pipeline([('preprocessor', logistic_pre),
                           ('classifier', LogisticRegression(max_iter=2000,
                                                             random_state=RANDOM_SEED))])
logistic_model.fit(X_train, y_train)

tree_pre = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL),
    ('cont', ____, CONTINUOUS)])

tree_model = Pipeline([('preprocessor', tree_pre),
                       ('classifier', DecisionTreeClassifier(max_depth=6, min_samples_leaf=50,
                                                             random_state=RANDOM_SEED))])
tree_model.fit(X_train, y_train)
print("both models fitted")

**Q5.2** Compare the three on the test set: accuracy, recall and precision at the default cutoff point of
0.5, and AUC. How many orders does each model flag as late at 0.5, and what is its recall? Why are so
few orders flagged? Cite the share of late orders in the data in your answer. Does this mean the
model is useless? Is there a way to still use the model?

In [ ]:
rows = []
for name, model in [('Baseline', baseline), ('Logistic regression', logistic_model),
                    ('Decision tree', tree_model)]:
    y_pred = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    rows.append({'model': name,
                 'accuracy': accuracy_score(y_test, y_pred),
                 'flagged at 0.5': int(y_pred.sum()),
                 'recall': recall_score(y_test, y_pred, zero_division=0),
                 'precision': precision_score(y_test, y_pred, zero_division=0),
                 'AUC': roc_auc_score(y_test, probs)})
pd.DataFrame(rows).round(3)

> 📊 **Slide 31** — the comparison table, and what it says about the default cutoff point of 0.5.

**Q5.3** How do the models fit? Is there overfitting? For each model, compare the AUC on the training set with the
AUC on the test set, and the late rate among its top-ranked orders (the top 5%, the same share as the
500 flags) on each set. Then fit a tree with **no** depth limit and compare its two AUCs. If the
training and test results are close, what does that say about how much can be predicted at the moment
of purchase?

In [ ]:
def top_share_late_rate(model, X_part, y_part, share=500 / len(X_test)):
    """Late rate among the top-ranked orders: the same share of orders as the 500 flags."""
    probs = model.predict_proba(X_part)[:, 1]
    k = int(round(share * len(X_part)))
    top = np.argsort(-probs, kind='stable')[:k]
    return y_part.values[top].mean()

rows = []
for name, model in [('Logistic regression', logistic_model), ('Decision tree', tree_model)]:
    rows.append({'model': name,
                 'AUC train': roc_auc_score(y_train, model.predict_proba(____)[:, 1]),
                 'AUC test': roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]),
                 'top 5% late rate, train': top_share_late_rate(model, X_train, y_train),
                 'top 5% late rate, test': top_share_late_rate(model, X_test, y_test)})
pd.DataFrame(rows).round(3)

In [ ]:
full_tree = Pipeline([('preprocessor', ColumnTransformer([
                          ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL),
                          ('cont', 'passthrough', CONTINUOUS)])),
                      ('classifier', DecisionTreeClassifier(random_state=RANDOM_SEED))])
full_tree.fit(X_train, y_train)

print("tree with no depth limit — leaves:", full_tree.named_steps['classifier'].get_n_leaves())
print(f"AUC train {roc_auc_score(y_train, full_tree.predict_proba(X_train)[:, 1]):.3f}   "
      f"test {roc_auc_score(y_test, full_tree.predict_proba(X_test)[:, 1]):.3f}")

> 📊 **Slide 32** — the training and test results for both models and for the tree with no depth
> limit, and your answer: overfitting, or little to predict at the moment of purchase?

---
## 6 · Using the capacity: the top 500

The team can act on 500 orders, so the useful question is not "is this order late?" but "which 500
orders are most likely to be late?" Rank the test orders by predicted probability and take the top 500.

**Q6.1** How many late orders does each model put in its top 500? Compare with the ~41 that 500
random flags would catch. What are recall and precision for the top 500?

In [ ]:
def top_k(model, k=500):
    probs = model.predict_proba(X_test)[:, 1]
    order = np.argsort(-probs, kind='stable')[:k]
    caught = int(y_test.values[order].sum())
    return {'late caught': caught, 'recall': caught / y_test.sum(),
            'precision': caught / ____, 'cutoff (lowest probability flagged)': probs[order[-1]]}

pd.DataFrame({'Logistic regression': top_k(logistic_model),
              'Decision tree': top_k(____)}).T.round(3)

> 📊 **Slide 33** — late orders caught by each model's top 500 against 500 random flags, with
> recall and precision.

**Q6.2** The sweep: for the model you would ship, try several cutoff points. For each, how many orders are
flagged, how many late orders are caught, recall, precision, and accuracy. What happens to recall, precision
and accuracy as the cutoff point changes? Why is accuracy the wrong measure for this problem?

In [ ]:
probs = ____.predict_proba(X_test)[:, 1]

rows = []
for cutoff in [0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.50]:
    pred = (probs >= cutoff).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    rows.append({'cutoff': cutoff, 'flagged': int(pred.sum()), 'late caught (TP)': tp,
                 'false alarms (FP)': fp, 'late missed (FN)': fn,
                 'recall': recall_score(y_test, pred, zero_division=0),
                 'precision': precision_score(y_test, pred, zero_division=0),
                 'accuracy': accuracy_score(y_test, pred)})
pd.DataFrame(rows).round(3)

> 📊 **Slide 34** — the sweep table, what happens to recall, precision and accuracy as the cutoff point
> changes, and why accuracy is the wrong measure for this problem.

---
## 7 · What drives the prediction

**Q7.1** For the logistic regression: which features push the probability of late delivery **up**,
and which push it **down**? A continuous coefficient is per standard deviation (the columns were
scaled), so compare continuous columns with each other, and categories with each other.

In [ ]:
names = logistic_model.named_steps['preprocessor'].get_feature_names_out()
coefs = pd.Series(logistic_model.named_steps['classifier'].coef_[0], index=names)

print("CONTINUOUS columns (per standard deviation), largest push up first:")
print(coefs[coefs.index.str.startswith('cont__')].sort_values(ascending=False).round(3), "\n")

categories = coefs[coefs.index.str.startswith('cat__')]
print("CATEGORIES that push UP the most:")
print(categories.sort_values(ascending=False).head(6).round(3), "\n")
print("CATEGORIES that push DOWN the most:")
print(categories.sort_values().head(6).round(3))

**Q7.2** For the decision tree: which features did most of the splitting (feature importances)?

In [ ]:
tree_names = tree_model.named_steps['preprocessor'].get_feature_names_out()
importances = pd.Series(tree_model.named_steps['classifier'].____, index=tree_names)
importances.sort_values(ascending=False).head(10).round(3)

> 📊 **Slide 35** — the top drivers from each model, in plain English. A coefficient says which way
> and how strongly a column pushes; an importance says how much of the splitting (dividing the orders into branches) it did in the tree. Say what each
> one tells about the order's delivery.

**Q7.3** Look inside the tree. How deep is it, and how many leaves (final groups of orders) does it
have? What does it check first? Which columns does it never use, even though they were offered as
features?

In [ ]:
tree = tree_model.named_steps['classifier']
tree_names = list(tree_model.named_steps['preprocessor'].get_feature_names_out())
print("depth:", tree.get_depth(), "   leaves:", tree.get_n_leaves(), "\n")
print(export_text(tree, feature_names=tree_names, max_depth=____))

used = [name for name, importance in zip(tree_names, tree.feature_importances_) if importance > 0]
never_used = [col for col in CONTINUOUS + CATEGORICAL
              if not any(name == f'cont__{col}' or name.startswith(f'cat__{col}_') for name in used)]
print("columns the tree never splits on:", never_used)

**Q7.4** Each leaf is a rule: a chain of conditions, and the late rate of the training orders that
meet them. The code lists the four leaves with the highest late rate in the training set, their late
rate on the test set, and the conditions that lead to each. Write each rule in plain English —
`cat__customer_state_SP <= 0.50` means the customer is **not** in SP. How many orders does each rule
rest on? Does its late rate hold up on the test set? Which rule would you give the fulfillment
team, and why?

In [ ]:
X_train_tree = tree_model.named_steps['preprocessor'].transform(X_train)
X_test_tree = tree_model.named_steps['preprocessor'].transform(X_test)
train_leaves = pd.DataFrame({'leaf': tree.apply(X_train_tree), 'late': y_train.values})
test_leaves = pd.DataFrame({'leaf': tree.apply(X_test_tree), 'late': y_test.values})
leaf_table = (train_leaves.groupby('leaf')['late'].agg(n_orders='count', late_rate='mean')
              .join(test_leaves.groupby('leaf')['late'].agg(n_orders_test='count', late_rate_test='mean'))
              .fillna(0).sort_values('late_rate', ascending=____))

def leaf_rule(leaf_id):
    """The chain of conditions from the top of the tree down to one leaf."""
    t = tree.tree_
    parent = {}
    for node in range(t.node_count):
        if t.children_left[node] != -1:
            parent[t.children_left[node]] = (node, '<=')
            parent[t.children_right[node]] = (node, '>')
    conditions = []
    node = leaf_id
    while node in parent:
        node, sign = parent[node]
        conditions.append(f"{tree_names[t.feature[node]]} {sign} {t.threshold[node]:.2f}")
    return conditions[::-1]

for leaf_id in leaf_table.index[:4]:
    row = leaf_table.loc[leaf_id]
    print(f"leaf {leaf_id}: training {int(row.n_orders)} orders, {row.late_rate:.1%} late  |  "
          f"test {int(row.n_orders_test)} orders, {row.late_rate_test:.1%} late")
    for condition in leaf_rule(leaf_id):
        print("     ", condition)

> 📊 **Slide 36** — the tree's depth and leaves, what it checks first, the columns it never uses, and
> the four riskiest rules in plain English with their training and test late rates.

---
## 8 · Is the model worth using?

An AUC of 0.71 would not impress a data scientist, and most models used in business are no better.
The fulfillment team's question is different: **do the flags earn more than they cost?**

**Q8.1** Why does a late order cost the company anything? Ask your Notebook 2 teammate for the
review comparison of late and on-time orders (Notebook 2, Q6.1, slide 21): the mean review score and
the share of bad reviews in orders late and on-time. What else does a late order cost? Think about contacts to
customer support, vouchers or refunds, and customers who do not order again.

**Q8.2** Estimate the cost on the late order flags. Each flag costs money: a proactive message with a voucher, or
paying for faster shipping. A caught late order saves money only if the action prevents the harm:
the parcel arrives on time, or the customer is told in advance and does not leave a bad review.

The three numbers at the top of the following cell are **assumptions of estimated costs**, not data. Change them to values your
team can defend, and say why. Compare the model's 500 flags with 500 random flags, then find the
number of flags that earns the most. The **break-even saving** is the potential saving per caught late order at
which the flags exactly pay for themselves; above it, flagging earns money.

In [ ]:
COST_PER_FLAG = 10                            # BRL (Brazilian Real) per flagged order, late or not:
                                              # a proactive message with a voucher, or faster shipping
POTENTIAL_SAVING_PER_CAUGHT_LATE_ORDER = 100  # BRL saved if the action fully works: support contact,
                                              # voucher or refund, lost repeat business avoided
ACTION_SUCCESS_RATE = 0.5                     # share of caught late orders where the action works

def net_value(late_caught, n_flags):
    saving = late_caught * ACTION_SUCCESS_RATE * POTENTIAL_SAVING_PER_CAUGHT_LATE_ORDER
    return saving - n_flags * ____

probs = logistic_model.predict_proba(X_test)[:, 1]
ranked_late = y_test.values[np.argsort(-probs, kind='stable')]

rows = []
for n_flags in [100, 200, 300, 500, 750, 1000, 1500, 2000]:
    caught = int(ranked_late[:n_flags].sum())
    rows.append({'flags': n_flags, 'late caught': caught,
                 'precision': round(caught / n_flags, 3),
                 'net value, model (BRL)': round(net_value(caught, n_flags)),
                 'net value, random flags (BRL)': round(net_value(n_flags * y_test.mean(), n_flags))})
print(pd.DataFrame(rows).to_string(index=False), "\n")

# Break-even: the POTENTIAL_SAVING_PER_CAUGHT_LATE_ORDER at which the flags exactly pay for themselves.
# Set net value to 0 and solve for the saving:
#   late_caught * ACTION_SUCCESS_RATE * saving - n_flags * COST_PER_FLAG = 0
def break_even_saving(late_caught, n_flags):
    return n_flags * COST_PER_FLAG / (late_caught * ACTION_SUCCESS_RATE)

caught_500 = int(ranked_late[:500].sum())
print(f"assumed potential saving per caught late order: {POTENTIAL_SAVING_PER_CAUGHT_LATE_ORDER} BRL")
print(f"break-even, 500 model flags:   {break_even_saving(caught_500, 500):.0f} BRL")
print(f"break-even, 500 random flags:  {break_even_saving(500 * y_test.mean(), 500):.0f} BRL")

**Q8.3** The fulfillment manager says: "An AUC of 0.71 sounds weak. Why should we use this?"
Answer in two or three sentences, with your numbers: what the model does that random flags cannot,
how many orders you would flag, and which assumption, if it were wrong, would make the model not worth
using.

> 📊 **Slide 37** — the review numbers from Notebook 2, your assumptions, the value of the model's
> flags against random flags, the number of flags you would use, and your answer to the manager.

---
## 9 · Recommendation, and how to do better

**Q9.1** Would you recommend that the fulfillment team use this model to choose which orders to flag?
Use your numbers from Q8.2: how many more late orders it catches than random flags, what it earns, how
many orders you would flag, and what it misses.

**Q9.2** How could the model be improved? Think about more than the algorithm: different data,
different features, and **when** the prediction is made. If you test an idea, use the cell below to run your code and
report what happened.

In [ ]:
# Optional: test an improvement idea here

> 📊 **Slide 38** — your recommendation, and your improvement ideas.

---
### Before you submit

- [ ] Runtime → **Restart and run all** finishes with no errors
- [ ] Every 📊 slide above (27–38) is filled in the deck
- [ ] Your finding for slide 39, the team summary, is agreed with your team
- [ ] File → Download → **Download .ipynb**, and upload that file to Canvas. Download *after* the run above, so the file includes every output